In [24]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [25]:
import numpy as np
import pandas as pd

from algorithms.baseline import UserBasedCF
from metrics.metrics import ndcg_at_k, recall_at_k, test_matrix_info
from utils.data_load import load_dataset, make_subset, save_df, split_train_test_by_time
from utils.timer import timer


In [26]:
DATA_PATH = "../data/raw/ratings.csv"

DATA_TRAIN_1M_PATH = "../data/processed/train_1M.csv"
DATA_TEST_1M_PATH = "../data/processed/test_1M.csv"

In [27]:
df = load_dataset(path=DATA_PATH)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB


In [28]:
df.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [29]:
try:
    train = load_dataset(DATA_TRAIN_1M_PATH)
    test = load_dataset(DATA_TEST_1M_PATH)
except FileNotFoundError:
    subset = make_subset(df=df, n_ratings=1_000_000)
    train, test = split_train_test_by_time(subset)
    save_df(df=train, path=DATA_TRAIN_1M_PATH)
    save_df(df=test, path=DATA_TEST_1M_PATH)

print(train.shape, test.shape)

(800500, 4) (200086, 4)


In [30]:
train.describe()

,userId,movieId,rating,timestamp
count,800500.000000,800500.000000,800500.000000,8.005000e+05
mean,98646.270209,25929.397535,3.572943,1.266412e+09
std,58418.514135,46291.101800,1.062505,2.560993e+08
min,141.000000,1.000000,0.500000,8.243892e+08
25%,47598.000000,1209.000000,3.000000,1.031770e+09
50%,98500.000000,3081.000000,4.000000,1.254978e+09
75%,150270.000000,33166.000000,4.000000,1.498865e+09
max,200935.000000,290263.000000,5.000000,1.696168e+09


In [31]:
test.describe()

,userId,movieId,rating,timestamp
count,200086.00000,200086.000000,200086.000000,2.000860e+05
mean,98647.76692,41712.255165,3.481840,1.306111e+09
std,58423.08884,64574.528694,1.059222,2.653764e+08
min,141.00000,1.000000,0.500000,8.271370e+08
25%,47598.00000,1517.000000,3.000000,1.070599e+09
50%,98484.00000,4487.000000,3.500000,1.335776e+09
75%,150291.00000,65802.000000,4.000000,1.546649e+09
max,200935.00000,292141.000000,5.000000,1.697157e+09


In [32]:
def build_test_matrix(model, test):
    if not hasattr(model, "user_to_index") or not hasattr(model, "item_to_index"):
        return None
    if not hasattr(model, "train_matrix"):
        return None

    test_matrix = np.zeros_like(model.train_matrix)
    for row in test.itertuples(index=False):
        if row.userId in model.user_to_index and row.movieId in model.item_to_index:
            u = model.user_to_index[row.userId]
            i = model.item_to_index[row.movieId]
            test_matrix[u, i] = row.rating
    return test_matrix

In [ ]:
def run_benchmark(
    model,
    train: pd.DataFrame,
    test: pd.DataFrame,
    k: int = 10,
    threshold: float = 3.5,
):

    with timer(f"Обучение {model.name}"):
        model.fit(train)

    model.info()

    test_users = test[model.user_id_col].unique().tolist()
    with timer(f"Выдача рекомендаций {model.name}"):
        recs = model.recommend(test_users, n=k)
    
    pred_matrix = model.get_pred_matrix()
    test_matrix = build_test_matrix(model, test)

    if pred_matrix is not None and test_matrix is not None:
        recall = recall_at_k(pred_matrix, test_matrix, k=k, threshold=threshold)
        ndcg = ndcg_at_k(pred_matrix, test_matrix, k=k)
        print(f"Recall@{k} = {recall}")
        print(f"NDCG@{k} = {ndcg}")
        test_matrix_info(test_matrix)

In [34]:
model = UserBasedCF(top_k=20)

run_benchmark(model, train, test)

[timer] Обучение UserBasedCF: 2.469 сек.
Количество пользователей: 6,314
Количество объектов: 21,388
Количество оценок в train: 800,500
Разреженность матрицы оценок: 0.5928%
Размер матрицы схожестей: 0.15 ГБ
[timer] Выдача рекомендаций UserBasedCF: 5.273 сек.
Recall@10 = 0.02803864435793312
NDCG@10 = 0.02464104623284104
Количество ненулевых оценок в тестовой матрице: 194,773
Количество релевантных оценок в тестовой матрице (>=3.5): 120,582
Пользователей с ненулевыми оценками в тестовой матрице: 6,313
Пользователей с релевантными оценками в тестовой матрице: 6,215
